In [17]:
import pandas as pd
import numpy as np

## 컬럼명 변경 규칙 (COLUMN_MAP)

In [18]:
# 컬럼명 변경 규칙
# - 컬럼명에서는 단위 제거
# - 공정 약어는 유지해 팀원 간 의사소통 가능하게 구성

COLUMN_MAP = {
    # 1) 시간·식별 변수
    "Batch_ID": "배치번호",
    "Batch ref": "배치번호참조",
    "Time (h)": "발효시간",

    # 2) 핵심 품질·공정 상태
    "Penicillin concentration(P:g/L)": "페니실린농도_P",
    "Substrate concentration(S:g/L)": "기질농도_S",
    "Dissolved oxygen concentration(DO2:mg/L)": "용존산소_DO2",
    "pH(pH:pH)": "pH",
    "Temperature(T:K)": "발효온도_T",
    "Vessel Volume(V:L)": "발효조부피_V",
    "Vessel Weight(Wt:Kg)": "발효조중량_Wt",

    # 3) 조작·투입·제어 변수
    "Agitator RPM(RPM:RPM)": "교반속도_RPM",
    "Sugar feed rate(Fs:L/h)": "당공급유량_Fs",
    "Aeration rate(Fg:L/h)": "공기주입유량_Fg",
    "Acid flow rate(Fa:L/h)": "산투입유량_Fa",
    "Base flow rate(Fb:L/h)": "염기투입유량_Fb",
    "Heating/cooling water flow rate(Fc:L/h)": "냉난방수유량_Fc",
    "Heating water flow rate(Fh:L/h)": "가열수유량_Fh",
    "Water for injection/dilution(Fw:L/h)": "희석수유량_Fw",
    "PAA flow(Fpaa:PAA flow (L/h))": "PAA투입유량_Fpaa",
    "Oil flow(Foil:L/hr)": "오일투입유량_Foil",
    "Ammonia shots(NH3_shots:kgs)": "암모니아투입량_NH3",

    # 4) 가스·대사·에너지 및 배출 변수
    "Air head pressure(pressure:bar)": "발효조상부압력",
    "carbon dioxide percent in off-gas(CO2outgas:%)": "배가스CO2_CO2out",
    "Oxygen in percent in off-gas(O2:O2  (%))": "배가스O2_O2out",
    "Oxygen Uptake Rate(OUR:(g min^{-1}))": "산소소비율_OUR",
    "Carbon evolution rate(CER:g/h)": "이산화탄소발생률_CER",   # <- 수정: _Cout -> _CER로 통일
    "Generated heat(Q:kJ)": "발생열량_Q",
    "Dumped broth flow(Fremoved:L/h)": "배양액제거유량_Fremoved",

    # 5) 결함 및 원본 제어 라벨
    "Fault ref(0-NoFault 1-Fault)": "결함라벨_배치",
    "Fault reference(Fault_ref:Fault ref)": "결함라벨_구간",
    "0 - Recipe driven 1 - Operator controlled(Control_ref:Control ref)": "원본제어플래그",

    # 6) 오프라인 측정 변수
    "Offline Biomass concentratio(X_offline:X(g L^{-1}))": "바이오매스농도_오프라인",
    "Offline Penicillin concentration(P_offline:P(g L^{-1}))": "페니실린농도_오프라인",
    "PAA concentration offline(PAA_offline:PAA (g L^{-1}))": "PAA농도_오프라인",
    "NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))": "암모니아농도_오프라인",
    "Viscosity(Viscosity_offline:centPoise)": "점도_오프라인",

    # 7) 배치 단위 생산 KPI
    "Penicllin_harvested_during_batch(kg)": "중간수확량",
    "Penicllin_harvested_end_of_batch (kg)": "종료수확량",
    "Penicllin_yield_total (kg)": "총회수량",
}

## 단위 사전 (시각화용)

In [19]:
# 단위 사전 - 컬럼명에는 단위를 넣지 않고 시각화할 때만 사용

UNIT_MAP = {
    "발효시간": "h",
    "페니실린농도_P": "g/L",
    "기질농도_S": "g/L",
    "용존산소_DO2": "mg/L",
    "pH": None,
    "발효온도_T": "K",
    "발효조부피_V": "L",
    "발효조중량_Wt": "kg",
    "교반속도_RPM": "RPM",
    "당공급유량_Fs": "L/h",
    "공기주입유량_Fg": "L/h",
    "산투입유량_Fa": "L/h",
    "염기투입유량_Fb": "L/h",
    "냉난방수유량_Fc": "L/h",
    "가열수유량_Fh": "L/h",
    "희석수유량_Fw": "L/h",
    "PAA투입유량_Fpaa": "L/h",
    "오일투입유량_Foil": "L/h",
    "암모니아투입량_NH3": "kg",
    "발효조상부압력": "bar",
    "배가스CO2_CO2out": "%",
    # 원본 컬럼명은 %지만 실제 값은 fraction일 가능성이 있어 확인 필요
    "배가스O2_O2out": "원본 표기 %",
    "산소소비율_OUR": "g/min",
    "이산화탄소발생률_CER": "g/h",
    "발생열량_Q": "kJ",
    "배양액제거유량_Fremoved": "L/h",
    "바이오매스농도_오프라인": "g/L",
    "페니실린농도_오프라인": "g/L",
    "PAA농도_오프라인": "g/L",
    "암모니아농도_오프라인": "g/L",
    "점도_오프라인": "cP",
    "중간수확량": "kg",
    "종료수확량": "kg",
    "총회수량": "kg",
}

## 요인별 컬럼 정렬 순서

In [20]:
# 요인별 컬럼 정렬 순서

COLUMN_GROUPS = {
    "식별·시간": ["배치번호", "제어전략그룹", "발효시간"],
    "핵심 품질·공정 상태": [
        "페니실린농도_P", "기질농도_S", "용존산소_DO2", "pH",
        "발효온도_T", "발효조부피_V", "발효조중량_Wt",
    ],
    "조작·투입·제어": [
        "교반속도_RPM", "당공급유량_Fs", "공기주입유량_Fg", "산투입유량_Fa",
        "염기투입유량_Fb", "냉난방수유량_Fc", "가열수유량_Fh", "희석수유량_Fw",
        "PAA투입유량_Fpaa", "오일투입유량_Foil", "암모니아투입량_NH3",
    ],
    "가스·대사·에너지": [
        "발효조상부압력", "배가스CO2_CO2out", "배가스O2_O2out",
        "산소소비율_OUR", "이산화탄소발생률_CER", "발생열량_Q", "배양액제거유량_Fremoved",
    ],
    "결함·원본 라벨": ["원본제어플래그", "결함라벨_구간", "결함라벨_배치"],
    "오프라인 측정": [
        "바이오매스농도_오프라인", "페니실린농도_오프라인",
        "PAA농도_오프라인", "암모니아농도_오프라인", "점도_오프라인",
    ],
    "배치 KPI": ["중간수확량", "종료수확량", "총회수량"],
}

## 데이터 정리 함수 정의

In [21]:
# 데이터 정리 함수

def prepare_merged_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    merged_data의 컬럼명을 변경하고, 제어전략그룹을 생성한 뒤,
    요인별 순서로 컬럼을 재정렬합니다.
    """
    df = df.copy()

    # 원본 컬럼과 매핑 사전 일치 여부 확인
    unmapped_columns = sorted(set(df.columns) - set(COLUMN_MAP))
    nonexistent_columns = sorted(set(COLUMN_MAP) - set(df.columns))

    if unmapped_columns:
        raise ValueError(f"COLUMN_MAP에 등록되지 않은 원본 컬럼이 있습니다:\n{unmapped_columns}")
    if nonexistent_columns:
        raise ValueError(f"현재 데이터에 존재하지 않는 컬럼이 COLUMN_MAP에 있습니다:\n{nonexistent_columns}")

    # Batch_ID와 Batch ref 중복 여부 검증
    batch_columns_equal = df["Batch_ID"].equals(df["Batch ref"])
    if not batch_columns_equal:
        mismatch_count = (df["Batch_ID"] != df["Batch ref"]).sum()
        raise ValueError(f"Batch_ID와 Batch ref 값이 일치하지 않습니다. 불일치 행 수: {mismatch_count:,}")

    df = df.drop(columns=["Batch ref"])

    rename_map = {k: v for k, v in COLUMN_MAP.items() if k != "Batch ref"}
    df = df.rename(columns=rename_map)

    # Batch_ID 범위 기준 제어전략그룹 생성
    batch_id = df["배치번호"]
    invalid_batch = ~batch_id.between(1, 100)
    if invalid_batch.any():
        invalid_values = sorted(batch_id.loc[invalid_batch].unique())
        raise ValueError(f"배치번호 범위를 벗어난 값이 있습니다: {invalid_values}")

    conditions = [
        batch_id.between(1, 30),
        batch_id.between(31, 60),
        batch_id.between(61, 90),
        batch_id.between(91, 100),
    ]
    choices = ["RC", "OC", "APC", "Fault"]

    # Fault는 제어전략이 아닌 평가 그룹이므로 '제어전략'이 아닌 '제어전략그룹'으로 지정
    batch_group = np.select(conditions, choices, default="Unknown")
    df.insert(loc=1, column="제어전략그룹", value=batch_group)

    # 정의한 그룹 순서대로 컬럼 재정렬
    ordered_columns = [col for group_cols in COLUMN_GROUPS.values() for col in group_cols]

    missing_order_columns = [col for col in ordered_columns if col not in df.columns]
    if missing_order_columns:
        raise ValueError(f"정렬 목록에는 있지만 데이터에 없는 컬럼이 있습니다:\n{missing_order_columns}")

    # 혹시 새 파생변수가 추가됐을 경우 맨 뒤에 보존
    unassigned_columns = [col for col in df.columns if col not in ordered_columns]
    df = df[ordered_columns + unassigned_columns]

    return df

## 실행 및 확인

In [22]:
df = pd.read_csv("merged_data.csv")   # 실제 파일 경로로 수정
df_clean = prepare_merged_data(df)

print("정리 완료:", df_clean.shape)
print(df_clean.columns.tolist())

정리 완료: (113935, 39)
['배치번호', '제어전략그룹', '발효시간', '페니실린농도_P', '기질농도_S', '용존산소_DO2', 'pH', '발효온도_T', '발효조부피_V', '발효조중량_Wt', '교반속도_RPM', '당공급유량_Fs', '공기주입유량_Fg', '산투입유량_Fa', '염기투입유량_Fb', '냉난방수유량_Fc', '가열수유량_Fh', '희석수유량_Fw', 'PAA투입유량_Fpaa', '오일투입유량_Foil', '암모니아투입량_NH3', '발효조상부압력', '배가스CO2_CO2out', '배가스O2_O2out', '산소소비율_OUR', '이산화탄소발생률_CER', '발생열량_Q', '배양액제거유량_Fremoved', '원본제어플래그', '결함라벨_구간', '결함라벨_배치', '바이오매스농도_오프라인', '페니실린농도_오프라인', 'PAA농도_오프라인', '암모니아농도_오프라인', '점도_오프라인', '중간수확량', '종료수확량', '총회수량']


## 저장 (선택)

In [24]:
df_clean.to_csv("merged_data_clean.csv", index=False, encoding="utf-8-sig")